# Handle data to SQL

## 1. Imports

In [46]:
import pandas as pd
import os # Para poder buscar las variables de entorno
from dotenv import load_dotenv # Cargar variables de entorno
from sqlalchemy import create_engine

## 2. Import data

In [47]:
df = pd.read_csv('../data/raw/train.csv')
df.tail()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
9795,9796,CA-2017-125920,21/05/2017,28/05/2017,Standard Class,SH-19975,Sally Hughsby,Corporate,United States,Chicago,Illinois,60610.0,Central,OFF-BI-10003429,Office Supplies,Binders,"Cardinal HOLDit! Binder Insert Strips,Extra St...",3.798
9796,9797,CA-2016-128608,12/01/2016,17/01/2016,Standard Class,CS-12490,Cindy Schnelling,Corporate,United States,Toledo,Ohio,43615.0,East,OFF-AR-10001374,Office Supplies,Art,"BIC Brite Liner Highlighters, Chisel Tip",10.368
9797,9798,CA-2016-128608,12/01/2016,17/01/2016,Standard Class,CS-12490,Cindy Schnelling,Corporate,United States,Toledo,Ohio,43615.0,East,TEC-PH-10004977,Technology,Phones,GE 30524EE4,235.188
9798,9799,CA-2016-128608,12/01/2016,17/01/2016,Standard Class,CS-12490,Cindy Schnelling,Corporate,United States,Toledo,Ohio,43615.0,East,TEC-PH-10000912,Technology,Phones,Anker 24W Portable Micro USB Car Charger,26.376
9799,9800,CA-2016-128608,12/01/2016,17/01/2016,Standard Class,CS-12490,Cindy Schnelling,Corporate,United States,Toledo,Ohio,43615.0,East,TEC-AC-10000487,Technology,Accessories,SanDisk Cruzer 4 GB USB Flash Drive,10.384


## 3. Handle data to SQL

### 3.1 Orden inicial

Cambiar 'Order Date' a datetime para ordenar la información

In [48]:
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')

In [49]:
df = df.sort_values(by='Order Date', ascending=True)

In [50]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
7980,7981,CA-2015-103800,2015-01-03,07/01/2015,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,Texas,77095.0,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448
741,742,CA-2015-112326,2015-01-04,08/01/2015,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540.0,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540
740,741,CA-2015-112326,2015-01-04,08/01/2015,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540.0,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736
739,740,CA-2015-112326,2015-01-04,08/01/2015,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540.0,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784
1759,1760,CA-2015-141817,2015-01-05,12/01/2015,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,Pennsylvania,19143.0,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536


Eliminar 'Row ID'y resetear índice.

In [51]:
df = df.drop(columns=['Row ID'])

In [52]:
df.reset_index(inplace=True, drop=True)

In [53]:
df.head()

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,CA-2015-103800,2015-01-03,07/01/2015,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,Texas,77095.0,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448
1,CA-2015-112326,2015-01-04,08/01/2015,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540.0,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540
2,CA-2015-112326,2015-01-04,08/01/2015,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540.0,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736
3,CA-2015-112326,2015-01-04,08/01/2015,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540.0,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784
4,CA-2015-141817,2015-01-05,12/01/2015,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,Pennsylvania,19143.0,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536


### 3.2 Get each different product

In [54]:
df_products = df[['Product ID', 'Product Name', 'Category', 'Sub-Category']].drop_duplicates()
df_products.head()

,Product ID,Product Name,Category,Sub-Category
0,OFF-PA-10000174,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",Office Supplies,Paper
1,OFF-BI-10004094,GBC Standard Plastic Binding Systems Combs,Office Supplies,Binders
2,OFF-ST-10002743,SAFCO Boltless Steel Shelving,Office Supplies,Storage
3,OFF-LA-10003223,Avery 508,Office Supplies,Labels
4,OFF-AR-10003478,Avery Hi-Liter EverBold Pen Style Fluorescent ...,Office Supplies,Art


Cambiar nombres de las columnas para que coincidan con las de la base de datos.

In [33]:
df_products.columns = [
    'product_id',
    'product_name',
    'category',
    'subcategory'
]
df_products.head()

,product_id,product_name,category,subcategory
0,OFF-PA-10000174,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",Office Supplies,Paper
1,OFF-BI-10004094,GBC Standard Plastic Binding Systems Combs,Office Supplies,Binders
2,OFF-ST-10002743,SAFCO Boltless Steel Shelving,Office Supplies,Storage
3,OFF-LA-10003223,Avery 508,Office Supplies,Labels
4,OFF-AR-10003478,Avery Hi-Liter EverBold Pen Style Fluorescent ...,Office Supplies,Art


### 3.3 Get each different geography

In [34]:
df_geography = df[['Country', 'City', 'State', 'Region', 'Postal Code']].drop_duplicates()
df_geography.head()

,Country,City,State,Region,Postal Code
0,United States,Houston,Texas,Central,77095.0
1,United States,Naperville,Illinois,Central,60540.0
4,United States,Philadelphia,Pennsylvania,East,19143.0
5,United States,Henderson,Kentucky,South,42420.0
7,United States,Los Angeles,California,West,90049.0


Cambiar nombres para la base de datos.

In [35]:
df_geography.columns = [
    'country',
    'city',
    'state',
    'region',
    'postal_code'
]
df_geography.head()

,country,city,state,region,postal_code
0,United States,Houston,Texas,Central,77095.0
1,United States,Naperville,Illinois,Central,60540.0
4,United States,Philadelphia,Pennsylvania,East,19143.0
5,United States,Henderson,Kentucky,South,42420.0
7,United States,Los Angeles,California,West,90049.0


Considerar que aquí nos estaría faltando el id de cada geography

### 3.4 Get each different customer

In [36]:
df_customers = df[['Customer ID', 'Customer Name']].drop_duplicates()
df_customers.head()

,Customer ID,Customer Name
0,DP-13000,Darren Powers
1,PO-19195,Phillina Ober
4,MB-18085,Mick Brown
5,ME-17320,Maria Etezadi
7,LS-17230,Lycoris Saunders


Cambiar nombres de columnas.

In [37]:
df_customers.columns = ['customer_id', 'customer_name']
df_customers.head()

,customer_id,customer_name
0,DP-13000,Darren Powers
1,PO-19195,Phillina Ober
4,MB-18085,Mick Brown
5,ME-17320,Maria Etezadi
7,LS-17230,Lycoris Saunders


### 3.5 Get sales data

Ojo que aquí obtendremos cada registro de venta, luego se va a diferenciar con un id unico.

Además considerar que el geography_id queda pendiente, ya que debemos generarlo primero en la tabla geography.

In [38]:
df.head()

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,CA-2015-103800,2015-01-03,07/01/2015,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,Texas,77095.0,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448
1,CA-2015-112326,2015-01-04,08/01/2015,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540.0,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540
2,CA-2015-112326,2015-01-04,08/01/2015,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540.0,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736
3,CA-2015-112326,2015-01-04,08/01/2015,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540.0,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784
4,CA-2015-141817,2015-01-05,12/01/2015,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,Pennsylvania,19143.0,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536


In [39]:
df_sales = df[['Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Segment', 'Product ID', 'Customer ID', 'Sales']].drop_duplicates()
df_sales.head()

,Order ID,Order Date,Ship Date,Ship Mode,Segment,Product ID,Customer ID,Sales
0,CA-2015-103800,2015-01-03,07/01/2015,Standard Class,Consumer,OFF-PA-10000174,DP-13000,16.448
1,CA-2015-112326,2015-01-04,08/01/2015,Standard Class,Home Office,OFF-BI-10004094,PO-19195,3.540
2,CA-2015-112326,2015-01-04,08/01/2015,Standard Class,Home Office,OFF-ST-10002743,PO-19195,272.736
3,CA-2015-112326,2015-01-04,08/01/2015,Standard Class,Home Office,OFF-LA-10003223,PO-19195,11.784
4,CA-2015-141817,2015-01-05,12/01/2015,Standard Class,Consumer,OFF-AR-10003478,MB-18085,19.536


Cambiar nombres de columnas.

In [40]:
df_sales.columns = ['order_id', 'order_date', 'ship_date', 'ship_mode', 'segment', 'product_id', 'customer_id', 'sales']
df_sales.head()

,order_id,order_date,ship_date,ship_mode,segment,product_id,customer_id,sales
0,CA-2015-103800,2015-01-03,07/01/2015,Standard Class,Consumer,OFF-PA-10000174,DP-13000,16.448
1,CA-2015-112326,2015-01-04,08/01/2015,Standard Class,Home Office,OFF-BI-10004094,PO-19195,3.540
2,CA-2015-112326,2015-01-04,08/01/2015,Standard Class,Home Office,OFF-ST-10002743,PO-19195,272.736
3,CA-2015-112326,2015-01-04,08/01/2015,Standard Class,Home Office,OFF-LA-10003223,PO-19195,11.784
4,CA-2015-141817,2015-01-05,12/01/2015,Standard Class,Consumer,OFF-AR-10003478,MB-18085,19.536


## 4. Connect to database

Para este punto debemos tener una base de datos con las tablas creadas.

Cargar y guardar variables de entorno.

In [41]:
load_dotenv()

True

In [42]:
user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST')
port = os.getenv('DB_PORT')
database = os.getenv('DB_NAME')


Crear engine. Esto sera la conexión a nuestra base de datos.

In [43]:
engine = create_engine(
    f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}'
)

## 5. Load data into database

Productos

In [44]:
df_products.head()

,product_id,product_name,category,subcategory
0,OFF-PA-10000174,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",Office Supplies,Paper
1,OFF-BI-10004094,GBC Standard Plastic Binding Systems Combs,Office Supplies,Binders
2,OFF-ST-10002743,SAFCO Boltless Steel Shelving,Office Supplies,Storage
3,OFF-LA-10003223,Avery 508,Office Supplies,Labels
4,OFF-AR-10003478,Avery Hi-Liter EverBold Pen Style Fluorescent ...,Office Supplies,Art


In [ ]:
df_products.to_sql(
    'product',          # Nombre de la tabla
    engine,             # Conexión a la base de datos
    if_exists='append', # Agregar datos si la tabla ya existe
    index=False         # No añadir como columna el indice del DataFrame
)

Geography (ojo que aquí al cargarse, como no se le envió un id)

In [ ]:
df_geography.head()

,country,city,state,region,postal_code
0,United States,Houston,Texas,Central,77095.0
1,United States,Naperville,Illinois,Central,60540.0
4,United States,Philadelphia,Pennsylvania,East,19143.0
5,United States,Henderson,Kentucky,South,42420.0
7,United States,Los Angeles,California,West,90049.0


In [ ]:
df_geography.to_sql(
    'geography',
    engine,
    if_exists='append',
    index=False
)

Customer

In [ ]:
df_customers.head()

,customer_id,customer_name
0,DP-13000,Darren Powers
1,PO-19195,Phillina Ober
4,MB-18085,Mick Brown
5,ME-17320,Maria Etezadi
7,LS-17230,Lycoris Saunders


In [ ]:
df_customers.to_sql(
    'customer',
    engine,
    if_exists='append',
    index=False
)